In [ ]:
!pip install segmentation-models-pytorch albumentations

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 6.4 MB/s eta 0:00:00


In [ ]:
import os
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import albumentations as A
from albumentations.pytorch import ToTensorV2

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
IMAGE_DIR = "/content/drive/MyDrive/skindataset/train_images_processed"
MASK_DIR  = "/content/drive/MyDrive/skindataset/Training_GroundTruth"

image_paths = sorted([os.path.join(IMAGE_DIR, i) for i in os.listdir(IMAGE_DIR)])
mask_paths = sorted([os.path.join(MASK_DIR, i) for i in os.listdir(MASK_DIR)])

train_imgs, val_imgs, train_masks, val_masks = train_test_split(
    image_paths, mask_paths, test_size=0.2, random_state=42
)

In [ ]:
train_transform = A.Compose([
    A.Resize(256,256),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Normalize(),
    ToTensorV2()
], is_check_shapes=False)

val_transform = A.Compose([
    A.Resize(256,256),
    A.Normalize(),
    ToTensorV2()
], is_check_shapes=False)

In [ ]:
class SkinDataset(Dataset):
    def __init__(self, images, masks, transform=None):
        self.images = images
        self.masks = masks
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        image = cv2.imread(self.images[idx])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(self.masks[idx],0)
        mask = np.expand_dims(mask,axis=-1)

        if self.transform:
            aug = self.transform(image=image, mask=mask)
            image = aug["image"]
            mask = aug["mask"].float() / 255.0 # Normalize mask values to [0, 1]
            mask = mask.permute(2,0,1)

        return image, mask

In [ ]:
train_dataset = SkinDataset(train_imgs, train_masks, train_transform)
val_dataset = SkinDataset(val_imgs, val_masks, val_transform)

train_loader = DataLoader(train_dataset,batch_size=8,shuffle=True)
val_loader = DataLoader(val_dataset,batch_size=8,shuffle=False)

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self,in_c,out_c):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c,out_c,3,1,1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c,out_c,3,1,1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True)
        )
    def forward(self,x):
        return self.conv(x)

In [ ]:
class UNet(nn.Module):

    def __init__(self):
        super().__init__()

        self.down1 = DoubleConv(3,64)
        self.down2 = DoubleConv(64,128)
        self.down3 = DoubleConv(128,256)
        self.down4 = DoubleConv(256,512)

        self.pool = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(512,1024)

        self.up1 = nn.ConvTranspose2d(1024,512,2,2)
        self.conv1 = DoubleConv(1024,512)

        self.up2 = nn.ConvTranspose2d(512,256,2,2)
        self.conv2 = DoubleConv(512,256)

        self.up3 = nn.ConvTranspose2d(256,128,2,2)
        self.conv3 = DoubleConv(256,128)

        self.up4 = nn.ConvTranspose2d(128,64,2,2)
        self.conv4 = DoubleConv(128,64)

        self.out = nn.Conv2d(64,1,1)


    def forward(self,x):

        d1 = self.down1(x)
        p1 = self.pool(d1)

        d2 = self.down2(p1)
        p2 = self.pool(d2)

        d3 = self.down3(p2)
        p3 = self.pool(d3)

        d4 = self.down4(p3)
        p4 = self.pool(d4)

        bottleneck = self.bottleneck(p4)

        up1 = self.up1(bottleneck)
        up1 = torch.cat([up1,d4],dim=1)
        up1 = self.conv1(up1)

        up2 = self.up2(up1)
        up2 = torch.cat([up2,d3],dim=1)
        up2 = self.conv2(up2)

        up3 = self.up3(up2)
        up3 = torch.cat([up3,d2],dim=1)
        up3 = self.conv3(up3)

        up4 = self.up4(up3)
        up4 = torch.cat([up4,d1],dim=1)
        up4 = self.conv4(up4)

        return torch.sigmoid(self.out(up4))

In [ ]:
def dice_coef(pred, target, smooth=1):

    pred = pred.view(-1)
    target = target.view(-1)

    intersection = (pred * target).sum()

    return (2.*intersection + smooth) / (pred.sum() + target.sum() + smooth)

In [ ]:
def iou_score(pred,target,smooth=1):

    pred = pred.view(-1)
    target = target.view(-1)

    intersection = (pred * target).sum()

    total = (pred + target).sum()
    union = total - intersection

    return (intersection + smooth)/(union + smooth)

In [ ]:
def pixel_accuracy(pred,target):

    pred = (pred > 0.5).float()

    correct = (pred == target).float()

    return correct.sum()/correct.numel()

In [ ]:
def precision_recall(pred,target):

    pred = (pred > 0.5).float()

    TP = (pred*target).sum()
    FP = (pred*(1-target)).sum()
    FN = ((1-pred)*target).sum()

    precision = TP/(TP+FP+1e-8)
    recall = TP/(TP+FN+1e-8)

    return precision, recall

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = UNet().to(device)

criterion = nn.BCELoss()

optimizer = optim.Adam(model.parameters(),lr=1e-4)

In [ ]:
epochs = 30

for epoch in range(epochs):

    model.train()
    train_loss = 0

    for images,masks in train_loader:

        images = images.to(device)
        masks = masks.to(device)

        preds = model(images)

        loss = criterion(preds,masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    print("Epoch:",epoch+1,"Loss:",train_loss/len(train_loader))

Epoch: 1 Loss: 0.4104863660203086
Epoch: 2 Loss: 0.307211147248745
Epoch: 3 Loss: 0.26830449567900766
Epoch: 4 Loss: 0.2527064484026697
Epoch: 5 Loss: 0.2257139296995269
Epoch: 6 Loss: 0.21614140305254195
Epoch: 7 Loss: 0.2030058690243297
Epoch: 8 Loss: 0.18499038815498353
Epoch: 9 Loss: 0.18243562016222212
Epoch: 10 Loss: 0.17100368870629204
Epoch: 11 Loss: 0.16623833262258106
Epoch: 12 Loss: 0.17209456952081786
Epoch: 13 Loss: 0.1616909915374385
Epoch: 14 Loss: 0.15633132192823623
Epoch: 15 Loss: 0.1559318749441041
Epoch: 16 Loss: 0.15000716828637653
Epoch: 17 Loss: 0.1442921332187123
Epoch: 18 Loss: 0.1438463605940342
Epoch: 19 Loss: 0.13736685179173946
Epoch: 20 Loss: 0.13735328473978572
Epoch: 21 Loss: 0.1383975659807523
Epoch: 22 Loss: 0.13664996268020735
Epoch: 23 Loss: 0.13221249373422728
Epoch: 24 Loss: 0.13559137392375203
Epoch: 25 Loss: 0.12476483682791392
Epoch: 26 Loss: 0.1247853713730971
Epoch: 27 Loss: 0.12986319458319082
Epoch: 28 Loss: 0.13981937799188826
Epoch: 29 Los

In [ ]:
epochs = 30

for epoch in range(epochs):

    model.val()
    val_loss = 0

    for images,masks in val_loader:

        images = images.to(device)
        masks = masks.to(device)

        preds = model(images)

        loss = criterion(preds,masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        val_loss += loss.item()

    print("Epoch:",epoch+1,"Loss:",val_loss/len(val_loader))

In [ ]:
model.eval()

dice_total = 0
iou_total = 0
acc_total = 0
precision_total = 0
recall_total = 0

with torch.no_grad():

    for images,masks in val_loader:

        images = images.to(device)
        masks = masks.to(device)

        preds = model(images)

        dice_total += dice_coef(preds,masks)
        iou_total += iou_score(preds,masks)
        acc_total += pixel_accuracy(preds,masks)

        precision, recall = precision_recall(preds,masks)

        precision_total += precision
        recall_total += recall

print("Dice:", dice_total/len(val_loader))
print("IoU:", iou_total/len(val_loader))
print("Pixel Accuracy:", acc_total/len(val_loader))
print("Precision:", precision_total/len(val_loader))
print("Recall:", recall_total/len(val_loader))